# SLM-SAM2 Thigh Segmentation — Water (Lambda Cloud)

Runs [mazurowski-lab/SLM-SAM2](https://github.com/mazurowski-lab/SLM-SAM2) on **Dixon WATER** NIfTI stacks
using **Dafne water segmentations as mask prompts** — same pipeline as
`muscle_map_wb+slmsam/codes/slm_sam2_lambda_water.ipynb`, with the prompt
source swapped from MuscleMap WB to Dafne.

The SAM2 video predictor is initialised with a single annotated slice (middle of each
muscle's extent in the Dafne mask) and propagated bidirectionally through the volume.

**Designed for a Lambda Cloud GPU instance (A10 or similar) — this does not run
locally.** Same as the MuscleMap WB version, this notebook must be uploaded to
and executed on a GPU cloud instance.

## Prerequisites
Upload to Lambda before running:
- Water NIfTI stacks at `~/myosegmenTUM/` (subject folders with `ImageData/*_WATER/`)
- Dafne water segmentations at `~/dafne_water_segs/` (files named
  `*_dafne_thigh.npz`, one file per subject/stack, each containing one
  binary mask array per muscle key e.g. `Gracilis_R`, `Sartorius_L` — this is
  the same format already produced in `dafne/segs_water/`)

```bash
# water images
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/myosegmenTUM \
  your machine9:~/

# Dafne water segmentations (source: dafne/segs_water on this machine)
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/dafne/segs_water/ \
  your machine9:~/dafne_water_segs/
```

## Steps
1. Launch a Lambda instance and open JupyterLab
2. Upload this notebook to `~`
3. Run all cells top to bottom
4. Results saved to `~/dafne_slm_sam2_segs_water/`
5. Download results:
```bash
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  your machine9:~/dafne_slm_sam2_segs_water/ \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/dafne+slmsam/segs_water/
```
6. **Terminate the instance when done**
7. Then run `eval_notebooks/dafne+slmsam/codes/column_compare_dafne_slmsam2.ipynb`
   locally to score the downloaded results against ground truth, and re-run
   `eval_notebooks/morr_graphs/with_sam.ipynb` to add the new bar.

In [ ]:
import subprocess, sys, os

REPO_DIR = '/home/ubuntu/SLM-SAM2'

if not os.path.isdir(REPO_DIR):
    subprocess.check_call(['git', 'clone',
        'https://github.com/mazurowski-lab/SLM-SAM2.git', REPO_DIR])
    print('Cloned SLM-SAM2')
else:
    print('SLM-SAM2 already cloned')

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
    print('Added to sys.path:', REPO_DIR)

# Pin numpy<2 first — the pre-installed torch build on Lambda's base image is
# compiled against NumPy 1.x, and none of hydra-core/iopath/omegaconf pin
# numpy themselves, so letting pip resolve freely can upgrade to 2.x and
# break torch's ABI compatibility at import/inference time.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--user', '-q',
                        'numpy<2'])

# The sam2 package uses Hydra for its config system (build_sam2_video_predictor
# loads a .yaml via hydra), plus iopath/omegaconf as its other core deps.
# These aren't pulled in just by cloning the repo, so install them explicitly
# alongside the packages this notebook itself needs.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--user', '-q',
                        'hydra-core', 'iopath', 'omegaconf',
                        'SimpleITK', 'Pillow'])
print('Dependencies ready')

In [ ]:
import subprocess, os

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

result = subprocess.run(['nvidia-smi',
    '--query-compute-apps=pid,used_memory,process_name',
    '--format=csv,noheader'], capture_output=True, text=True)
print('Processes using GPU:')
print(result.stdout or '  (none)')

mem = subprocess.run(['nvidia-smi', '--query-gpu=memory.free,memory.total',
                      '--format=csv,noheader,nounits'],
                     capture_output=True, text=True)
free, total = [int(x) for x in mem.stdout.strip().split(', ')]
print(f'\nGPU memory: {free} MiB free / {total} MiB total')
if free < 4000:
    print('\nWARNING: less than 4 GB free.')
    print('Shut down other notebook kernels in JupyterLab before continuing.')

In [ ]:
import os, urllib.request

CKPT_DIR  = '/home/ubuntu/SLM-SAM2/checkpoints'
CKPT_FILE = os.path.join(CKPT_DIR, 'sam2.1_hiera_tiny.pt')
URL = 'https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_tiny.pt'

os.makedirs(CKPT_DIR, exist_ok=True)
if not os.path.exists(CKPT_FILE):
    print('Downloading checkpoint...')
    urllib.request.urlretrieve(URL, CKPT_FILE)
    print(f'Done ({os.path.getsize(CKPT_FILE) // 1_000_000} MB)')
else:
    print('Checkpoint already present')

In [ ]:
import os, glob, shutil, tempfile
import numpy as np
import torch
import SimpleITK as sitk
from PIL import Image

DAFNE_SEGS_DIR = os.path.expanduser('~/dafne_water_segs')
OUTPUT_DIR     = os.path.expanduser('~/dafne_slm_sam2_segs_water')
CHECKPOINT     = '/home/ubuntu/SLM-SAM2/checkpoints/sam2.1_hiera_tiny.pt'
MODEL_CFG      = 'configs/sam2.1/slm_sam2_hiera_t.yaml'

IMAGE_GLOB = os.path.expanduser('~/myosegmenTUM/*/ImageData/*_WATER/*_WATER_stack*.nii')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Device:               ', DEVICE)
print('DAFNE_SEGS_DIR exists:', os.path.isdir(DAFNE_SEGS_DIR))
image_files = sorted(glob.glob(IMAGE_GLOB))
print(f'Found {len(image_files)} water stacks')

In [ ]:
from sam2.build_sam import build_sam2_video_predictor

torch.autocast(device_type='cuda', dtype=torch.bfloat16).__enter__()
if torch.cuda.is_available() and torch.cuda.get_device_properties(0).major >= 8:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

predictor = build_sam2_video_predictor(MODEL_CFG, CHECKPOINT)
predictor.eval()
print('SLM-SAM2 predictor loaded on', DEVICE)

In [ ]:
def export_slices_as_jpg(img_array, out_dir):
    """Export (D, H, W) float array as numbered RGB JPG files for SAM2 video input."""
    os.makedirs(out_dir, exist_ok=True)
    for i in range(img_array.shape[0]):
        sl = img_array[i]
        sl_norm  = (sl - sl.min()) / (sl.max() - sl.min() + 1e-8)
        sl_uint8 = (sl_norm * 255).astype(np.uint8)
        rgb = np.stack([sl_uint8] * 3, axis=-1)
        Image.fromarray(rgb).save(os.path.join(out_dir, f'{i}.jpg'))


def find_prompt_slice(muscle_volume):
    """Return the slice index closest to the centre of the muscle's axial extent."""
    slices_with_mask = np.where(muscle_volume.any(axis=(1, 2)))[0]
    if len(slices_with_mask) == 0:
        return None
    return int(slices_with_mask[len(slices_with_mask) // 2])


print('Helper functions defined')

In [ ]:
# Dafne's water npz files are named '{subject}_WATER_stack{N}_dafne_thigh.npz'
# and already contain one binary (D, H, W) mask per muscle key (e.g. 'Gracilis_R'),
# unlike MuscleMap's single multi-label .nii.gz — no integer LABEL_MAP needed here.
matched, missing = [], []
for nii_path in image_files:
    stem     = os.path.splitext(os.path.basename(nii_path))[0]
    seg_path = os.path.join(DAFNE_SEGS_DIR, f'{stem}_dafne_thigh.npz')
    if os.path.exists(seg_path):
        matched.append((nii_path, seg_path))
    else:
        missing.append(stem)

print(f'Matched: {len(matched)}  Missing Dafne seg: {len(missing)}')
if missing:
    print('  Missing:', missing[:5], '...' if len(missing) > 5 else '')

In [ ]:
MUSCLE_BATCH_SIZE = 5

for nii_path, seg_path in matched:
    stem     = os.path.splitext(os.path.basename(nii_path))[0]
    out_path = os.path.join(OUTPUT_DIR, f'{stem}_dafne_slmsam2.npz')

    if os.path.exists(out_path):
        print(f'Skipping (done): {stem}')
        continue

    print(f'\nProcessing: {stem}')

    img_sitk  = sitk.ReadImage(nii_path)
    img_array = sitk.GetArrayFromImage(img_sitk).astype(float)  # (D, H, W)
    print(f'  Image shape: {img_array.shape}')

    dafne_data = np.load(seg_path)
    num_slices = img_array.shape[0]

    # build per-muscle prompts directly from Dafne's per-muscle binary masks
    muscle_prompts = {}
    for obj_id, muscle_name in enumerate(sorted(dafne_data.files), start=1):
        muscle_vol = dafne_data[muscle_name].astype(bool)
        if muscle_vol.shape != img_array.shape:
            print(f'  shape mismatch for {muscle_name}: '
                  f'{muscle_vol.shape} vs image {img_array.shape}, skipping muscle')
            continue
        prompt_slice = find_prompt_slice(muscle_vol)
        if prompt_slice is None:
            continue
        muscle_prompts[obj_id] = (muscle_name, prompt_slice,
                                  muscle_vol[prompt_slice].astype(np.uint8))

    if not muscle_prompts:
        print('  No muscles found in Dafne seg — skipping')
        continue

    print(f'  Muscles with prompts: {len(muscle_prompts)}/{len(dafne_data.files)}')

    tmp_dir   = tempfile.mkdtemp(prefix='dafne_slmsam2_')
    all_masks = {}

    try:
        export_slices_as_jpg(img_array, tmp_dir)

        items   = list(muscle_prompts.items())
        batches = [items[i:i+MUSCLE_BATCH_SIZE]
                   for i in range(0, len(items), MUSCLE_BATCH_SIZE)]

        for batch_idx, batch in enumerate(batches):
            print(f'  Batch {batch_idx+1}/{len(batches)}: '
                  f'{[v[0] for _, v in batch]}')

            torch.cuda.empty_cache()
            inference_state = predictor.init_state(video_path=tmp_dir)

            first_prompt = min(v[1] for _, v in batch)

            for obj_id, (muscle_name, prompt_slice, mask_2d) in batch:
                predictor.add_new_mask(
                    inference_state=inference_state,
                    frame_idx=prompt_slice,
                    obj_id=obj_id,
                    mask=mask_2d,
                )

            video_segments = {}

            for frame_idx, obj_ids, mask_logits in predictor.propagate_in_video(
                    inference_state, start_frame_idx=first_prompt, recent_n=1):
                video_segments[frame_idx] = {
                    oid: (mask_logits[i] > 0).cpu().numpy()
                    for i, oid in enumerate(obj_ids)
                }

            if first_prompt > 0:
                for frame_idx, obj_ids, mask_logits in predictor.propagate_in_video(
                        inference_state, start_frame_idx=first_prompt,
                        reverse=True, recent_n=1):
                    video_segments[frame_idx] = {
                        oid: (mask_logits[i] > 0).cpu().numpy()
                        for i, oid in enumerate(obj_ids)
                    }

            for obj_id, (muscle_name, _, _) in batch:
                vol_mask = np.zeros(img_array.shape, dtype=np.uint8)
                for slice_idx in range(num_slices):
                    seg_frame = video_segments.get(slice_idx, {})
                    if obj_id in seg_frame:
                        vol_mask[slice_idx] = np.squeeze(
                            seg_frame[obj_id]).astype(np.uint8)
                all_masks[muscle_name] = vol_mask

            predictor.reset_state(inference_state)
            torch.cuda.empty_cache()

        np.savez_compressed(out_path, **all_masks)
        print(f'  Saved -> {out_path}')

        dafne_voxels = {name: int(dafne_data[name].sum()) for name in dafne_data.files}
        print(f'  {"Muscle":<30} {"Dafne input":>12} {"Refined":>10}')
        print(f'  {"-"*54}')
        for name, vol in sorted(all_masks.items()):
            refined_v = int(vol.sum())
            input_v   = dafne_voxels.get(name, 0)
            flag      = '  <-- EMPTY' if refined_v == 0 and input_v > 0 else ''
            print(f'  {name:<30} {input_v:>12,} {refined_v:>10,}{flag}')

    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)

print('\nAll done.')

In [ ]:
results = sorted(glob.glob(os.path.join(OUTPUT_DIR, '*.npz')))
print(f'Total output files: {len(results)}')
if results:
    sample = np.load(results[0])
    print(f'Sample: {results[0]}')
    print(f'  {"Muscle":<30} {"Shape"} {"Voxels":>10}')
    for name in sorted(sample.files):
        arr = sample[name]
        print(f'  {name:<30} {str(arr.shape):<15} {int(arr.sum()):>10,}')